# Sales

In [ ]:
import os
import re
from bs4 import BeautifulSoup
import pandas as pd


def get_natural_sort_key(filename):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", filename)
    ]


def extract_sales_data(folder_path, clean_data_path):
    os.makedirs(clean_data_path, exist_ok=True)

    raw_files = [
        f for f in os.listdir(folder_path) if f.lower().endswith((".html", ".htm"))
    ]
    files = sorted(raw_files, key=get_natural_sort_key)

    print(f"Found {len(files)} HTML files.")

    all_rows = []

    for file in files:
        file_path = os.path.join(folder_path, file)

        try:
            with open(
                file_path, "r", encoding="windows-1256", errors="ignore"
            ) as f:
                soup = BeautifulSoup(f.read(), "lxml")

            rows = soup.find_all("tr")

            for row in rows:
                cols = row.find_all("td")
                cols_text = [c.get_text(strip=True) for c in cols]

                if len(cols_text) == 9 and "/" in cols_text[1]:
                    data_row = {
                        "Source_File": file,
                        "Date": cols_text[1],
                        "Invoice_No": cols_text[2],
                        "Item_Code": cols_text[3],
                        "Barcode": cols_text[4] if cols_text[4] != "" else None,
                        "Item_Name": cols_text[5],
                        "Price": cols_text[6],
                        "Quantity": cols_text[7],
                        "Total_Value": cols_text[8],
                    }
                    all_rows.append(data_row)

            print(f"Processed: {file}")

        except Exception as e:
            print(f"Error processing {file}: {e}")

    if all_rows:
        df = pd.DataFrame(all_rows)
        df = df.drop_duplicates()

        numeric_columns = ["Price", "Quantity", "Total_Value"]
        for col in numeric_columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        output_file = os.path.join(clean_data_path, "Sales_All.csv")
        df.to_csv(output_file, index=False, encoding="utf-8-sig")

        print("\nProcessing complete.")
        print(f"Total rows extracted: {len(df)}")
        print(f"Saved to: {output_file}")
    else:
        print("No valid sales data found.")


if __name__ == "__main__":
    SOURCE_FOLDER = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\Html Cottonil Files\المبيعات"
    OUTPUT_FOLDER = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\prepared data\Sales"

    extract_sales_data(SOURCE_FOLDER, OUTPUT_FOLDER)

Found 288 HTML files
✔ Successfully Processed: SLSCARD08.html
✔ Successfully Processed: SLSCARD08Page2.html
✔ Successfully Processed: SLSCARD08Page3.html
✔ Successfully Processed: SLSCARD08Page4.html
✔ Successfully Processed: SLSCARD08Page5.html
✔ Successfully Processed: SLSCARD08Page6.html
✔ Successfully Processed: SLSCARD08Page7.html
✔ Successfully Processed: SLSCARD08Page8.html
✔ Successfully Processed: SLSCARD08Page9.html
✔ Successfully Processed: SLSCARD08Page10.html
✔ Successfully Processed: SLSCARD08Page11.html
✔ Successfully Processed: SLSCARD08Page12.html
✔ Successfully Processed: SLSCARD08Page13.html
✔ Successfully Processed: SLSCARD08Page14.html
✔ Successfully Processed: SLSCARD08Page15.html
✔ Successfully Processed: SLSCARD08Page16.html
✔ Successfully Processed: SLSCARD08Page17.html
✔ Successfully Processed: SLSCARD08Page18.html
✔ Successfully Processed: SLSCARD08Page19.html
✔ Successfully Processed: SLSCARD08Page20.html
✔ Successfully Processed: SLSCARD08Page21.html
✔ Succ

# Purchases

In [ ]:
import os
import re
from bs4 import BeautifulSoup
import pandas as pd


def extract_page_number(filename):
    match = re.search(r"Page(\d+)", filename, re.IGNORECASE)
    return int(match.group(1)) if match else 1


def process_purchases_data(folder_path, output_file):
    files = sorted(
        [f for f in os.listdir(folder_path) if f.lower().endswith(".html")],
        key=extract_page_number,
    )

    print(f"Found {len(files)} HTML files to process.")

    all_final_rows = []
    current_doc_rows = []

    current_header = {
        "doc_no": "",
        "doc_date": "",
        "supplier": "",
        "store": "المخزن الرئيسي",
        "keeper": "أمين غير محدد",
    }

    for file in files:
        path = os.path.join(folder_path, file)

        try:
            with open(path, "r", encoding="windows-1256", errors="ignore") as f:
                html_content = f.read()

            soup = BeautifulSoup(html_content, "html.parser")
            tables = soup.find_all("table")
            full_text = soup.get_text(" ", strip=True)

            has_new_header = "رقم إذن" in full_text or "المورد" in full_text

            if has_new_header:
                if current_doc_rows:
                    all_final_rows.extend(current_doc_rows)
                    current_doc_rows = []

                current_header = {
                    "doc_no": "",
                    "doc_date": "",
                    "supplier": "",
                    "store": "المخزن الرئيسي",
                    "keeper": "أمين غير محدد",
                }

                for tbl in tables[:4]:
                    text = tbl.get_text(" ", strip=True)

                    if "رقم إذن" in text or "إذن" in text:
                        no_match = re.search(r"رقم\s*إذن\s*(\d+)", text)
                        if not no_match:
                            no_match = re.search(r"\d+", text)
                        if no_match:
                            doc_id = (
                                no_match.group(1)
                                if len(no_match.groups()) > 0
                                else no_match.group(0)
                            )
                            current_header["doc_no"] = f"رقم إذن {doc_id}"

                    if "202" in text or "التاريخ" in text:
                        date_match = re.search(r"\d{4}/\d{1,2}/\d{1,2}", text)
                        if date_match:
                            current_header["doc_date"] = date_match.group(0)

                    if "المورد" in text:
                        sup_match = re.search(
                            r"المورد\s*:\s*([\u0600-\u06FF]+)", text
                        )
                        if sup_match:
                            current_header["supplier"] = sup_match.group(1)

            has_footer = (
                "أمين" in full_text
                or "المخــزن" in full_text
                or "المخزن" in full_text
            )

            if has_footer:
                for tbl in tables:
                    tds = [
                        td.get_text(strip=True)
                        for td in tbl.find_all("td")
                        if td.get_text(strip=True)
                    ]

                    for i, val in enumerate(tds):
                        if ("أمين" in val) and i + 1 < len(tds):
                            next_val = tds[i + 1].replace(":", "").strip()
                            if (
                                next_val
                                and next_val
                                not in [
                                    "المخزن",
                                    "المخــزن",
                                    "غير محدد",
                                    "أمين",
                                ]
                                and not next_val.startswith("المخزن")
                            ):
                                current_header["keeper"] = (
                                    next_val
                                    if next_val.startswith("أمين")
                                    else f"أمين {next_val}"
                                )

                        if (
                            "المخزن" in val or "المخــزن" in val
                        ) and i + 1 < len(tds):
                            next_val = tds[i + 1].replace(":", "").strip()
                            if (
                                next_val
                                and next_val not in ["أمين", "غير محدد"]
                                and not next_val.startswith("أمين")
                            ):
                                current_header["store"] = (
                                    next_val
                                    if "مخزن" in next_val
                                    else f"المخزن {next_val}"
                                )

            for tbl in tables:
                rows_in_tbl = tbl.find_all("tr")

                for tr in rows_in_tbl:
                    tds = tr.find_all("td")

                    if len(tds) < 8:
                        continue

                    raw_values = [td.get_text(strip=True) for td in tds]

                    if any(
                        h in raw_values
                        for h in [
                            "الباركود",
                            "الكود",
                            "الصنف",
                            "الإجمالى",
                            "الشراء",
                            "اوكازيون",
                        ]
                    ):
                        continue

                    first_val = (
                        raw_values[0]
                        if raw_values[0] != ""
                        else (raw_values[1] if len(raw_values) > 1 else "")
                    )

                    if not (first_val.isdigit() and 3 <= len(first_val) <= 4):
                        continue

                    barcode = first_val
                    start_idx = 0 if raw_values[0] != "" else 1

                    item_name = (
                        raw_values[start_idx + 2]
                        if len(raw_values) > start_idx + 2
                        else ""
                    )
                    category = (
                        raw_values[start_idx + 3]
                        if len(raw_values) > start_idx + 3
                        else ""
                    )
                    size = (
                        raw_values[start_idx + 4]
                        if len(raw_values) > start_idx + 4
                        else ""
                    )
                    color = (
                        raw_values[start_idx + 5]
                        if len(raw_values) > start_idx + 5
                        else ""
                    )

                    if re.match(r"^\d+(\.\d+)?$", color):
                        color = "--"

                    price_buy = (
                        raw_values[start_idx + 6]
                        if len(raw_values) > start_idx + 6
                        else ""
                    )
                    price_retail = (
                        raw_values[start_idx + 7]
                        if len(raw_values) > start_idx + 7
                        else ""
                    )
                    discount = (
                        raw_values[start_idx + 8]
                        if len(raw_values) > start_idx + 8
                        else "0.00"
                    )
                    qty = (
                        raw_values[start_idx + 9]
                        if len(raw_values) > start_idx + 9
                        else ""
                    )

                    if item_name and barcode:
                        current_doc_rows.append(
                            {
                                "Source_File": file,
                                "Date": current_header["doc_date"],
                                "Doc_No": current_header["doc_no"],
                                "Supplier": current_header["supplier"],
                                "Barcode": barcode,
                                "Item_Name": item_name,
                                "Category": category,
                                "Size": size,
                                "Color": color,
                                "Buy_Price": price_buy,
                                "Retail_Price": price_retail,
                                "Discount": discount,
                                "Quantity": qty,
                                "Store": current_header["store"],
                                "Keeper": current_header["keeper"],
                            }
                        )

            for r in current_doc_rows:
                r["Keeper"] = current_header["keeper"]
                r["Store"] = current_header["store"]

            print(f"Processed: {file}")

        except Exception as e:
            print(f"Error processing {file}: {e}")

    if current_doc_rows:
        all_final_rows.extend(current_doc_rows)

    columns = [
        "Source_File",
        "Date",
        "Doc_No",
        "Supplier",
        "Barcode",
        "Item_Name",
        "Category",
        "Size",
        "Color",
        "Buy_Price",
        "Retail_Price",
        "Discount",
        "Quantity",
        "Store",
        "Keeper",
    ]

    df = pd.DataFrame(all_final_rows, columns=columns)
    df.drop_duplicates(inplace=True)

    numeric_cols = ["Buy_Price", "Retail_Price", "Discount", "Quantity"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"\nProcessing complete. Successfully extracted {len(df)} records.")


if __name__ == "__main__":
    SOURCE_FOLDER = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\Html Cottonil Files\المشتريات"
    OUTPUT_FILE = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\prepared data\Purchases\Purchases_Cleaned.csv"

    process_purchases_data(SOURCE_FOLDER, OUTPUT_FILE)

Found 367 HTML Files to process.
✔ Processed: prchcard12.html
✔ Processed: prchcard12Page2.html
✔ Processed: prchcard12Page3.html
✔ Processed: prchcard12Page4.html
✔ Processed: prchcard12Page5.html
✔ Processed: prchcard12Page6.html
✔ Processed: prchcard12Page7.html
✔ Processed: prchcard12Page8.html
✔ Processed: prchcard12Page9.html
✔ Processed: prchcard12Page10.html
✔ Processed: prchcard12Page11.html
✔ Processed: prchcard12Page12.html
✔ Processed: prchcard12Page13.html
✔ Processed: prchcard12Page14.html
✔ Processed: prchcard12Page15.html
✔ Processed: prchcard12Page16.html
✔ Processed: prchcard12Page17.html
✔ Processed: prchcard12Page18.html
✔ Processed: prchcard12Page19.html
✔ Processed: prchcard12Page20.html
✔ Processed: prchcard12Page21.html
✔ Processed: prchcard12Page22.html
✔ Processed: prchcard12Page23.html
✔ Processed: prchcard12Page24.html
✔ Processed: prchcard12Page25.html
✔ Processed: prchcard12Page26.html
✔ Processed: prchcard12Page27.html
✔ Processed: prchcard12Page28.html
✔

# Inventory

In [ ]:
import glob
import os
import re
from bs4 import BeautifulSoup
import pandas as pd

SIZE_PATTERN = re.compile(r"^(s|m|l|xl|xxl|3xl|4xl|5xl|\d+-\d+|\d+|\-\-)$", re.IGNORECASE)


def is_size_value(text):
    return bool(SIZE_PATTERN.match(text.strip()))


def process_inventory_data(input_folder, output_file):
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    files = sorted(glob.glob(os.path.join(input_folder, "*.html")))
    rows_data = []

    for file_path in files:
        file_name = os.path.basename(file_path)

        with open(file_path, "rb") as f:
            soup = BeautifulSoup(f, "html.parser")

        current_level = ""
        tables = soup.find_all("table")

        for table in tables:
            level_td = table.find(
                "td", string=lambda t: t and "طباعة الجرد على مستوى" in t
            )
            if level_td:
                next_tds = table.find_all("td")
                if len(next_tds) >= 4:
                    current_level = next_tds[-1].get_text(strip=True)
                continue

            tds = table.find_all("td")
            if not tds:
                continue

            raw_tds = [td.get_text(strip=True) for td in tds]
            combined_text = "".join(raw_tds)

            ignored_keywords = [
                "الصنف",
                "إجمالى",
                "طباعة بتاريخ",
                "قطونيل شيراتون",
            ]
            if not combined_text or any(k in combined_text for k in ignored_keywords):
                continue

            if len(tds) >= 7:
                first_val = tds[1].get_text(strip=True) if len(tds) > 1 else ""

                if first_val and is_size_value(first_val):
                    item_name = None
                    size = first_val
                    color = tds[2].get_text(strip=True) if len(tds) > 2 else ""
                    group = tds[3].get_text(strip=True) if len(tds) > 3 else ""
                else:
                    item_name = first_val if first_val else None
                    size = tds[2].get_text(strip=True) if len(tds) > 2 else ""
                    color = tds[3].get_text(strip=True) if len(tds) > 3 else ""
                    group = tds[4].get_text(strip=True) if len(tds) > 4 else ""

                balance = tds[-1].get_text(strip=True)

                barcode = ""
                for td in tds[4:-1]:
                    txt = td.get_text(strip=True)
                    if txt.isdigit():
                        barcode = txt
                        break

                if not barcode and len(tds) > 8:
                    barcode = tds[8].get_text(strip=True)

                rows_data.append(
                    {
                        "Source": file_name,
                        "Section": current_level,
                        "Item": item_name,
                        "Size": size,
                        "Colour": color,
                        "Group": group,
                        "Code": str(barcode).strip(),
                        "Balance": balance,
                    }
                )

    df = pd.DataFrame(rows_data)

    if not df.empty:
        df.replace({"--": "", "": None}, inplace=True)

        df["Item"] = df["Item"].ffill()
        df["Section"] = df["Section"].ffill()

        df.fillna("", inplace=True)

        for col in df.columns:
            df[col] = df[col].astype(str).str.strip()

        df["Balance"] = pd.to_numeric(df["Balance"], errors="coerce")

        df.reset_index(drop=True, inplace=True)
        df.to_csv(output_file, index=False, encoding="utf-8-sig")

        print(f"Data restructuring complete. Total records: {len(df)}")
        print(df.head(20).to_string())
    else:
        print("No valid inventory data found.")


if __name__ == "__main__":
    INPUT_FOLDER = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\Html Cottonil Files\جرد البضاعة"
    OUTPUT_FILE = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\prepared data\Products\Combined_Inventory.csv"

    process_inventory_data(INPUT_FOLDER, OUTPUT_FILE)

تم إعادة هيكلة البيانات بنجاح! إجمالي الأسطر: 849

         Source Section             Item   Size Colour  Group Code Balance
0   kbal25.html               فوطه 30*30                داخلي  291       1
1   kbal25.html          فوطه وسط 50*100                داخلي  292       8
2   kbal25.html             يشكير 70*140                داخلي  293       2
3   kbal25.html             بشكير 90*160                داخلي  294       7
4   kbal25.html              بشكير احرام                داخلي  295       3
5   kbal25.html  اطفالي     سالوبيت بيبي                داخلي  460       0
6   kbal25.html  اطفالي      كولون بناتي  11-12         داخلي  474       0
7   kbal25.html  اطفالي      كولون بناتي  13-14         داخلي  475       0
8   kbal25.html  اطفالي      كولون بناتي  15-16         داخلي  476       0
9   kbal25.html  اطفالي      كولون بناتي    3-4         داخلي  470       0
10  kbal25.html  اطفالي      كولون بناتي    5-6         داخلي  471       0
11  kbal25.html  اطفالي      كولون بناتي    7-8  

# Product Movement

In [ ]:
import os
import re
from bs4 import BeautifulSoup
import pandas as pd


def clean_text(text):
    if text:
        text = text.replace("\xa0", " ").strip()
        return re.sub(r"\s+", " ", text)
    return ""


def process_html_file(file_path):
    with open(file_path, "r", encoding="windows-1256", errors="ignore") as f:
        content = f.read()

    soup = BeautifulSoup(content, "html.parser")
    tables = soup.find_all("table")

    branch_name = ""
    supplier_name = ""
    file_name = os.path.basename(file_path)

    for table in tables[:3]:
        text = clean_text(table.get_text())
        if "قطونيل" in text:
            branch_name = text
        if "للمورد" in text:
            match = re.search(r"للمورد\s+([^\s]+)", text)
            if match:
                supplier_name = match.group(1)

    rows_data = []

    for table in tables:
        tds = table.find_all("td")

        if len(tds) in [11, 12]:
            values = [clean_text(td.get_text()) for td in tds]

            first_val = values[0] if values[0] != "" else values[1]
            ignored_keywords = ["الصنف", "الإجمالى", "طباعة", "المستهلك"]
            if any(keyword in first_val for keyword in ignored_keywords):
                continue

            if values[0] == "":
                values = values[1:]

            if len(values) == 11:
                row_dict = {
                    "Source_File": file_name,
                    "Branch": branch_name,
                    "Supplier": supplier_name,
                    "Item_Name": values[0],
                    "Retail_Price": values[1],
                    "Item_Code": values[2],
                    "Purchase": values[3],
                    "Purchase_Returns": values[4],
                    "Sales": values[5],
                    "Sales_Returns": values[6],
                    "Net_Product_Sales_QTY": values[7],
                    "Net_Product_Sales_Value": values[8],
                    "Balance_QTY": values[9],
                    "Balance_Value": values[10],
                }
                rows_data.append(row_dict)

    return rows_data


def process_all_files(input_dir, output_file):
    all_records = []

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    for file_name in os.listdir(input_dir):
        if file_name.lower().endswith((".html", ".htm")):
            file_path = os.path.join(input_dir, file_name)
            file_records = process_html_file(file_path)
            all_records.extend(file_records)

    if not all_records:
        print("No valid data found to process.")
        return

    df = pd.DataFrame(all_records)

    numeric_cols = [
        "Retail_Price",
        "Purchase",
        "Purchase_Returns",
        "Sales",
        "Sales_Returns",
        "Net_Product_Sales_QTY",
        "Net_Product_Sales_Value",
        "Balance_QTY",
        "Balance_Value",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"\nProcessing complete. Successfully saved {len(df)} records.")
    print(f"Saved to: {output_file}")


if __name__ == "__main__":
    SOURCE_DIRECTORY = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\Html Cottonil Files\حركه الاصناف\all"
    DESTINATION_FILE = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\prepared data\product movement\product_movement_clean.csv"

    process_all_files(SOURCE_DIRECTORY, DESTINATION_FILE)

### Validation

In [ ]:
import os
import re
from bs4 import BeautifulSoup
import pandas as pd


def clean_text(text):
    if text:
        text = text.replace("\xa0", " ")
        text = re.sub(r"\s+", " ", text)
        return text.strip()
    return ""


def extract_html_rows(file_path):
    rows = []

    with open(file_path, "r", encoding="windows-1256", errors="ignore") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    for table in soup.find_all("table"):
        tds = table.find_all("td")

        if len(tds) < 11:
            continue

        values = [clean_text(td.get_text()) for td in tds]

        if values and values[0] == "":
            values = values[1:]

        if len(values) < 11:
            continue

        first = values[0]
        ignored_headers = ["الصنف", "الإجمالى", "الاجمالى", "طباعة"]
        if first in ignored_headers:
            continue

        try:
            rows.append(
                {
                    "Source_File": os.path.basename(file_path),
                    "Item_Name": values[0],
                    "Retail_Price": pd.to_numeric(values[1], errors="coerce"),
                    "Item_Code": str(values[2]).strip(),
                    "Purchase": pd.to_numeric(values[3], errors="coerce"),
                    "Purchase_Returns": pd.to_numeric(
                        values[4], errors="coerce"
                    ),
                    "Sales": pd.to_numeric(values[5], errors="coerce"),
                    "Sales_Returns": pd.to_numeric(values[6], errors="coerce"),
                    "Net_Product_Sales_QTY": pd.to_numeric(
                        values[7], errors="coerce"
                    ),
                    "Net_Product_Sales_Value": pd.to_numeric(
                        values[8], errors="coerce"
                    ),
                    "Balance_QTY": pd.to_numeric(values[9], errors="coerce"),
                    "Balance_Value": pd.to_numeric(
                        values[10], errors="coerce"
                    ),
                }
            )
        except Exception:
            pass

    return pd.DataFrame(rows)


def run_validation_pipeline(html_folder, csv_file, report_file):
    if not os.path.exists(csv_file):
        print(f"Error: Target CSV file not found at {csv_file}")
        return

    csv_df = pd.read_csv(csv_file)

    summary = []
    missing = []

    for file in os.listdir(html_folder):
        if not file.lower().endswith((".html", ".htm")):
            continue

        file_path = os.path.join(html_folder, file)
        extracted_html_df = extract_html_rows(file_path)
        extracted_csv_df = csv_df[csv_df["Source_File"] == file]

        html_qty = extracted_html_df["Net_Product_Sales_QTY"].sum()
        csv_qty = extracted_csv_df["Net_Product_Sales_QTY"].sum()

        html_val = extracted_html_df["Net_Product_Sales_Value"].sum()
        csv_val = extracted_csv_df["Net_Product_Sales_Value"].sum()

        summary.append(
            {
                "File": file,
                "HTML Rows": len(extracted_html_df),
                "CSV Rows": len(extracted_csv_df),
                "Rows Match": len(extracted_html_df) == len(extracted_csv_df),
                "HTML Qty": html_qty,
                "CSV Qty": csv_qty,
                "Qty Match": round(html_qty, 2) == round(csv_qty, 2),
                "HTML Value": html_val,
                "CSV Value": csv_val,
                "Value Match": round(html_val, 2) == round(csv_val, 2),
            }
        )

        html_keys = set(
            zip(
                extracted_html_df["Item_Name"].astype(str),
                extracted_html_df["Item_Code"].astype(str),
                extracted_html_df["Balance_QTY"],
            )
        )

        csv_keys = set(
            zip(
                extracted_csv_df["Item_Name"].astype(str),
                extracted_csv_df["Item_Code"].astype(str),
                extracted_csv_df["Balance_QTY"],
            )
        )

        for row in html_keys - csv_keys:
            missing.append(
                {
                    "File": file,
                    "Item_Name": row[0],
                    "Item_Code": row[1],
                    "Balance_QTY": row[2],
                }
            )

    summary_df = pd.DataFrame(summary)
    missing_df = pd.DataFrame(missing)

    duplicate_df = csv_df[
        csv_df.duplicated(
            subset=[
                "Source_File",
                "Item_Name",
                "Item_Code",
                "Balance_QTY",
            ],
            keep=False,
        )
    ]

    os.makedirs(os.path.dirname(report_file), exist_ok=True)

    with pd.ExcelWriter(report_file) as writer:
        summary_df.to_excel(writer, index=False, sheet_name="Summary")
        missing_df.to_excel(writer, index=False, sheet_name="Missing Rows")
        duplicate_df.to_excel(writer, index=False, sheet_name="Duplicates")

    mismatched_files = len(summary_df[summary_df["Rows Match"] == False])

    print("\nValidation Summary Results:")
    print(summary_df.to_string(index=False))
    print(f"\nFiles Checked        : {len(summary_df)}")
    print(f"Files With Mismatches: {mismatched_files}")
    print(f"Missing Records      : {len(missing_df)}")
    print(f"Duplicate Records    : {len(duplicate_df)}")
    print(f"\nValidation report generated successfully:\n{report_file}")


if __name__ == "__main__":
    HTML_FOLDER = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\Html Cottonil Files\حركه الاصناف\all"
    CSV_FILE = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\prepared data\product movement\product_movement_clean.csv"
    REPORT_FILE = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\Validation_Report.xlsx"

    run_validation_pipeline(HTML_FOLDER, CSV_FILE, REPORT_FILE)

# Sales with discount

In [ ]:
import os
import re
from bs4 import BeautifulSoup
import pandas as pd


def extract_page_number(filename):
    match = re.search(r"Page(\d+)", filename, re.IGNORECASE)
    if match:
        return int(match.group(1))

    numbers = re.findall(r"\d+", filename)
    return int(numbers[-1]) if numbers else 0


def safe_float(val):
    if not val:
        return 0.0
    try:
        cleaned = str(val).replace(",", "").strip()
        return float(cleaned)
    except ValueError:
        return 0.0


def process_cottonil_sales_directory(
    input_folder,
    output_folder,
    csv_filename="all_sales_data.csv",
    report_filename="validation_report.txt",
):
    os.makedirs(output_folder, exist_ok=True)

    csv_output_path = os.path.join(output_folder, csv_filename)
    report_output_path = os.path.join(output_folder, report_filename)

    lines_to_output = []

    def log(text=""):
        print(text)
        lines_to_output.append(text)

    log("Starting Cottonil HTML sales data extraction processing pipeline...")

    if not os.path.exists(input_folder):
        log(f"Error: Input path does not exist: {input_folder}")
        return

    files = [
        f
        for f in os.listdir(input_folder)
        if f.lower().endswith((".html", ".htm"))
    ]
    files = sorted(files, key=extract_page_number)

    if not files:
        log("Error: No HTML files found in the specified directory.")
        return

    all_extracted_rows = []
    total_summary_pages = 0

    for file_name in files:
        file_path = os.path.join(input_folder, file_name)

        with open(file_path, "r", encoding="windows-1256", errors="ignore") as f:
            soup = BeautifulSoup(f.read(), "html.parser")

        page_text = soup.get_text()
        all_trs = soup.find_all("tr")

        has_permit_rows = False
        for row in all_trs:
            cols = [
                td.get_text(strip=True) for td in row.find_all(["td", "th"])
            ]
            if cols and cols[0].replace(",", "").strip().isdigit():
                has_permit_rows = True
                break

        if "التقرير النهائي" in page_text and not has_permit_rows:
            log(f"Summary page skipped: {file_name}")
            total_summary_pages += 1
            continue

        file_sales_count = 0

        for row in all_trs:
            tds = row.find_all(["td", "th"])
            cols = [td.get_text(strip=True) for td in tds]

            if not cols:
                continue

            clean_permit = cols[0].replace(",", "").strip()
            if not clean_permit.isdigit():
                continue

            permit_no = int(clean_permit)

            item_idx = -1
            for idx in range(1, len(cols)):
                cleaned = cols[idx].strip()
                if cleaned and re.search(r"[\u0600-\u06FFa-zA-Z]", cleaned):
                    item_idx = idx
                    break

            if item_idx == -1:
                continue

            item_name = cols[item_idx].strip()

            codes_between = [
                cols[i].strip() for i in range(1, item_idx) if cols[i].strip()
            ]
            barcode_val = codes_between[-1] if codes_between else ""

            numeric_cols = [
                c.strip() for c in cols[item_idx + 1 :] if c.strip() != ""
            ]

            price = (
                safe_float(numeric_cols[0]) if len(numeric_cols) > 0 else 0.0
            )
            qty = safe_float(numeric_cols[1]) if len(numeric_cols) > 1 else 0.0
            discount = (
                safe_float(numeric_cols[2]) if len(numeric_cols) > 2 else 0.0
            )
            taxes = (
                safe_float(numeric_cols[3]) if len(numeric_cols) > 3 else 0.0
            )
            net = safe_float(numeric_cols[4]) if len(numeric_cols) > 4 else 0.0
            val = safe_float(numeric_cols[5]) if len(numeric_cols) > 5 else 0.0

            size = numeric_cols[-2] if len(numeric_cols) >= 8 else "--"
            color = numeric_cols[-1] if len(numeric_cols) >= 8 else "--"

            all_extracted_rows.append(
                {
                    "Source_File": file_name,
                    "Permit_No": permit_no,
                    "Barcode": barcode_val,
                    "Item_Name": item_name,
                    "Price": price,
                    "Quantity": qty,
                    "Discount": discount,
                    "Taxes": taxes,
                    "Net": net,
                    "Value": val,
                    "Size": size,
                    "Color": color,
                    "Page_Num": extract_page_number(file_name),
                }
            )
            file_sales_count += 1

        log(f"Processed: {file_name} -> Extracted {file_sales_count} sales records.")

    df = pd.DataFrame(all_extracted_rows)

    if not df.empty:
        df = df.sort_values(by=["Page_Num", "Permit_No"]).drop(
            columns=["Page_Num"]
        )
        df.to_csv(csv_output_path, index=False, encoding="utf-8-sig")

        log("\nExecution Summary:")
        log(f"Total files processed : {len(files)}")
        log(f"Summary pages skipped : {total_summary_pages}")
        log(f"Total records extracted: {len(df)}")
        log(f"Output saved to       : {csv_output_path}")
    else:
        log("\nNo valid records extracted.")

    with open(report_output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines_to_output))


if __name__ == "__main__":
    INPUT_DIR = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\Html Cottonil Files\صافي المبيعات قبل وبعد الخصم"
    OUTPUT_DIR = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\prepared data\Sales_after_discount"

    process_cottonil_sales_directory(INPUT_DIR, OUTPUT_DIR)

# Suppliers Acc

In [ ]:
import glob
import os
import re
from bs4 import BeautifulSoup
import pandas as pd

INPUT_DIR = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\Html Cottonil Files\كشف حساب الموردين"
OUTPUT_DIR = r"D:\Hady Learning\Data\data analysis\Mazen Course\Real Data\Cottonil\prepared data\Suppliers"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "Cottonil_Suppliers_Statement.csv")


def parse_supplier_html(file_path):
    with open(file_path, "r", encoding="windows-1256", errors="ignore") as f:
        content = f.read()

    soup = BeautifulSoup(content, "html.parser")
    tables = soup.find_all("table")

    supplier_name = ""
    for t in tables[:4]:
        text = t.get_text()
        if "كشف حساب" in text:
            tds = t.find_all("td")
            for td in tds:
                val = td.get_text(strip=True)
                ignored_values = [
                    "كشف حساب",
                    "قطونيل شيراتون",
                    "للفترة من",
                    "إلى",
                    "رصيد بدايه",
                ]
                if val and val not in ignored_values:
                    if not re.search(r"\d", val):
                        supplier_name = val
                        break
            if supplier_name:
                break

    rows_data = []

    for i, t in enumerate(tables):
        tds = t.find_all("td")
        texts = [td.get_text(strip=True) for td in tds]

        if len(texts) >= 9 and any(
            re.search(r"\d{2}/\d{2}/\d{4}", txt) for txt in texts
        ):
            description = texts[1] if len(texts) > 1 else ""
            date = next(
                (
                    txt
                    for txt in texts
                    if re.search(r"\d{2}/\d{2}/\d{4}", txt)
                ),
                "",
            )

            invoices = texts[3] if len(texts) > 3 else "0.00"
            returns = texts[4] if len(texts) > 4 else "0.00"
            payments = texts[5] if len(texts) > 5 else "0.00"
            discount = texts[6] if len(texts) > 6 else "0.00"
            forwarded_balance = texts[7] if len(texts) > 7 else "0.00"
            balance = texts[8] if len(texts) > 8 else "0.00"

            raw_balance_type = texts[9] if len(texts) > 9 else ""
            if i + 1 < len(tables):
                next_tds = tables[i + 1].find_all("td")
                next_texts = [td.get_text(strip=True) for td in next_tds]
                if (
                    len(next_texts) >= 2
                    and next_texts[-1] in ["ئن", "ين", "دائن", "مدين"]
                ):
                    raw_balance_type += next_texts[-1]

            if any(k in raw_balance_type for k in ["دائن", "دا", "ئن"]):
                balance_type = "دائن"
            elif any(k in raw_balance_type for k in ["مدين", "مد", "ين"]):
                balance_type = "مدين"
            else:
                balance_type = raw_balance_type

            if "إستلام" in description or "استلام" in description:
                doc_type = "استلام"
            elif "ارتجاع" in description or "إرتجاع" in description:
                doc_type = "ارتجاع"
            elif "دفعة" in description:
                doc_type = "دفعة"
            else:
                doc_type = description.split()[0] if description else "أخرى"

            num_match = re.search(r"\d+", description)
            doc_number = num_match.group(0) if num_match else ""

            payment_term = "آجل" if "آجل" in description else "فوري"

            row_dict = {
                "file_name": os.path.basename(file_path),
                "supplier_name": supplier_name,
                "description": description,
                "doc_type": doc_type,
                "doc_number": doc_number,
                "payment_term": payment_term,
                "date": date,
                "invoices": invoices,
                "returns": returns,
                "payments": payments,
                "discount": discount,
                "forwarded_balance": forwarded_balance,
                "balance": balance,
                "balance_type": balance_type,
            }
            rows_data.append(row_dict)

    return rows_data


def get_natural_sort_key(s):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", s)
    ]


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    html_files = sorted(
        glob.glob(os.path.join(INPUT_DIR, "*.html")), key=get_natural_sort_key
    )

    if not html_files:
        print(f"Error: No HTML files found in directory:\n{INPUT_DIR}")
        return

    all_data = []

    for file_path in html_files:
        filename = os.path.basename(file_path)
        extracted_rows = parse_supplier_html(file_path)
        all_data.extend(extracted_rows)
        print(f"Processed: {filename} -> Extracted {len(extracted_rows)} records.")

    if all_data:
        df = pd.DataFrame(all_data)

        columns_order = [
            "file_name",
            "supplier_name",
            "description",
            "doc_type",
            "doc_number",
            "payment_term",
            "date",
            "invoices",
            "returns",
            "payments",
            "discount",
            "forwarded_balance",
            "balance",
            "balance_type",
        ]
        df = df[columns_order]

        numeric_columns = [
            "invoices",
            "returns",
            "payments",
            "discount",
            "forwarded_balance",
            "balance",
        ]
        for col in numeric_columns:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(",", ""), errors="coerce"
            ).fillna(0.0)

        df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
        print("\nProcessing completed successfully.")
        print(f"Total extracted records: {len(df)}")
        print(f"Output saved to: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()